# Scan & Batch Operations

This notebook demonstrates:

- **`scan()`** — one-command model overview that runs DLA, logit lens, attention,
  and attribution, then surfaces the most interesting findings
- **`batch()`** — run any operation over a dataset with automatic result aggregation
- **`trace_batch()` / `dla_batch()`** — convenience wrappers for common batch workflows

In [ ]:
import interpkit

model = interpkit.load("gpt2")

## Scan — One-Command Overview

The fastest way to understand what a model is doing on a given input.
Runs multiple analyses and produces a ranked summary of key findings.

In [ ]:
result = model.scan("The capital of France is")

### Inspecting scan results

The returned dict contains the full results from each sub-analysis.

In [ ]:
print("Analyses run:", [k for k in ('dla', 'lens', 'attention', 'attribution') if k in result])

if "prediction" in result:
    pred = result["prediction"]
    for tok, prob in zip(pred["top5_tokens"][:3], pred["top5_probs"][:3]):
        print(f"  {tok!r:12s} {prob:.1%}")

print(f"\n{len(result.get('findings', []))} key findings surfaced.")

### Exporting scan figures

Pass `save="prefix"` to export figures from each sub-analysis.

In [ ]:
# This creates scan_dla.png, scan_lens.png, scan_attn.png, scan_attr.png
# result = model.scan("The capital of France is", save="scan")

## Batch Operations

Run any InterpKit operation over a dataset of examples.
Results are collected per-example, and summary statistics are computed automatically.

### Batch causal tracing

Trace multiple clean/corrupted pairs and see which modules matter most *on average*.

In [ ]:
dataset = [
    {"clean": "The capital of France is", "corrupted": "The capital of Poland is"},
    {"clean": "The CEO of Apple is", "corrupted": "The CEO of Google is"},
    {"clean": "The Eiffel Tower is in Paris", "corrupted": "The Eiffel Tower is in Rome"},
]

results = model.trace_batch(dataset, top_k=10)

print(f"Processed {results['count']} examples")
if "summary" in results and "ranked_modules" in results["summary"]:
    print("\nTop modules by mean effect:")
    for entry in results["summary"]["ranked_modules"][:5]:
        print(f"  {entry['module']:40s}  mean={entry['mean_effect']:.3f}")

### Batch DLA

Run DLA over multiple inputs and see which components consistently contribute the most.

In [ ]:
texts = [
    "The capital of France is",
    "The capital of Germany is",
    "The largest city in Japan is",
    "The CEO of Apple is",
]

results = model.dla_batch(texts, top_k=5)

print(f"Processed {results['count']} examples")
if "summary" in results and "ranked_components" in results["summary"]:
    print("\nTop components by mean contribution:")
    for entry in results["summary"]["ranked_components"][:5]:
        print(f"  {entry['component']:12s}  mean={entry['mean_contribution']:+.4f}")

### Generic batch runner

The `batch()` method works with any operation. The dataset is a list of dicts —
each dict is unpacked as keyword arguments to the operation.

In [ ]:
# Batch activation patching
patch_dataset = [
    {"clean": "The capital of France is", "corrupted": "The capital of Poland is"},
    {"clean": "The Eiffel Tower is in Paris", "corrupted": "The Eiffel Tower is in Rome"},
]

results = model.batch(
    "patch",
    patch_dataset,
    op_kwargs={"at": "transformer.h.8.mlp"},
)

print(f"Mean patch effect: {results['summary']['mean_effect']:.3f}")
print(f"Std:  {results['summary']['std_effect']:.3f}")

### Error handling

If some examples fail (e.g. wrong input format), batch continues processing
the remaining examples and reports errors at the end.

In [ ]:
print(f"Errors: {len(results.get('errors', []))}")
for err in results.get("errors", []):
    print(f"  Example {err['index']}: {err['error']}")